## Example 1: joblib functionality test 1

In [ ]:
import joblib
import time
import math

# Example 1: Parallel class for parallel processing
print("Example 1: joblib.Parallel - Parallel Processing")

# Define a compute-intensive function
def square_number(n):
    time.sleep(0.1)  # Simulate computation
    return n * n

# Sequential processing
start_time = time.time()
sequential_results = [square_number(i) for i in range(10)]
sequential_time = time.time() - start_time
print(f"Sequential results: {sequential_results}")
print(f"Sequential time: {sequential_time:.2f} seconds")

# Parallel processing using Parallel class
start_time = time.time()
parallel_results = joblib.Parallel(n_jobs=2)(
    joblib.delayed(square_number)(i) for i in range(10)
)
parallel_time = time.time() - start_time
print(f"Parallel results: {parallel_results}")
print(f"Parallel time: {parallel_time:.2f} seconds")
print(f"Speedup: {sequential_time/parallel_time:.2f}x")

# Different backend demonstration
with joblib.parallel_backend('threading', n_jobs=3):
    threading_results = joblib.Parallel()(
        joblib.delayed(math.sqrt)(i) for i in range(1, 6)
    )
print(f"Threading backend results: {threading_results}")

# Verbose output
verbose_results = joblib.Parallel(n_jobs=2, verbose=1)(
    joblib.delayed(lambda x: x**3)(i) for i in range(3)
)
print(f"Verbose results: {verbose_results}")

print("Example 1 completed successfully\n")

## Example 2: joblib functionality test 2

In [ ]:
import joblib
import tempfile
import time
import os

# Example 2: Memory class for function caching
print("Example 2: joblib.Memory - Function Caching")

# Create temporary cache directory
cache_dir = tempfile.mkdtemp()
memory = joblib.Memory(cache_dir, verbose=0)

# Define expensive computation function
@memory.cache
def expensive_computation(n):
    print(f"Computing for n={n}...")
    time.sleep(0.5)  # Simulate expensive computation
    return n * n * n

# First call - will be cached
start_time = time.time()
result1 = expensive_computation(5)
first_call_time = time.time() - start_time
print(f"First call result: {result1}, Time: {first_call_time:.2f}s")

# Second call - will use cache
start_time = time.time()
result2 = expensive_computation(5)
second_call_time = time.time() - start_time
print(f"Second call result: {result2}, Time: {second_call_time:.4f}s")
print(f"Cache speedup: {first_call_time/second_call_time:.1f}x")

# Different input - new computation
result3 = expensive_computation(3)
print(f"Different input result: {result3}")

# Clear specific function cache
memory.clear()
print("Cache cleared")

# Manual cache control
@memory.cache
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

# Cached fibonacci computation
start_time = time.time()
fib_result = fibonacci(10)
fib_time = time.time() - start_time
print(f"Fibonacci(10) = {fib_result}, Time: {fib_time:.4f}s")

# Cleanup
import shutil
shutil.rmtree(cache_dir)
print("Example 2 completed successfully\n")

## Example 3: joblib functionality test 3

In [ ]:
import joblib
import tempfile
import numpy as np
import os

# Example 3: dump and load functions for efficient serialization
print("Example 3: joblib.dump and joblib.load - Efficient Serialization")

# Create sample data structures
data_dict = {
    'numbers': [1, 2, 3, 4, 5],
    'text': 'Hello joblib',
    'nested': {'a': 10, 'b': 20}
}

# Create numpy array
numpy_array = np.random.random((100, 50))

# Custom class
class DataProcessor:
    def __init__(self, name):
        self.name = name
        self.data = [i**2 for i in range(10)]
    
    def process(self):
        return sum(self.data)

processor = DataProcessor("test_processor")

# Create temporary file for serialization
with tempfile.NamedTemporaryFile(delete=False, suffix='.joblib') as tmp_file:
    temp_filename = tmp_file.name

# Serialize dictionary
joblib.dump(data_dict, temp_filename)
loaded_dict = joblib.load(temp_filename)
print(f"Serialized dict: {loaded_dict}")
print(f"Dict match: {data_dict == loaded_dict}")

# Serialize numpy array with compression
joblib.dump(numpy_array, temp_filename, compress=3)
loaded_array = joblib.load(temp_filename)
print(f"Array shape: {loaded_array.shape}")
print(f"Array match: {np.array_equal(numpy_array, loaded_array)}")

# Serialize custom object
joblib.dump(processor, temp_filename)
loaded_processor = joblib.load(temp_filename)
print(f"Loaded processor name: {loaded_processor.name}")
print(f"Processor result: {loaded_processor.process()}")

# Multiple objects in one file
combined_data = [data_dict, numpy_array, processor]
joblib.dump(combined_data, temp_filename)
loaded_combined = joblib.load(temp_filename)
print(f"Combined data length: {len(loaded_combined)}")

# Cleanup
os.unlink(temp_filename)
print("Example 3 completed successfully\n")

## Example 4: joblib functionality test 4

In [ ]:
import joblib
import time

# Example 4: delayed function for lazy evaluation
print("Example 4: joblib.delayed - Lazy Evaluation")

# Define functions for delayed execution
def add_numbers(a, b):
    print(f"Adding {a} + {b}")
    time.sleep(0.1)
    return a + b

def multiply_numbers(a, b):
    print(f"Multiplying {a} * {b}")
    time.sleep(0.1)
    return a * b

def power_number(base, exp):
    print(f"Computing {base}^{exp}")
    time.sleep(0.1)
    return base ** exp

# Create delayed objects (lazy evaluation)
delayed_add = joblib.delayed(add_numbers)(5, 3)
delayed_multiply = joblib.delayed(multiply_numbers)(4, 6)
delayed_power = joblib.delayed(power_number)(2, 8)

print("Delayed objects created (no computation yet)")
print(f"Delayed add object: {delayed_add}")

# Execute delayed computations using Parallel
results = joblib.Parallel(n_jobs=1)([delayed_add, delayed_multiply, delayed_power])
print(f"Executed results: {results}")
print(f"Add: {results[0]}, Multiply: {results[1]}, Power: {results[2]}")

# Combine delayed with Parallel for lazy parallel execution
delayed_tasks = [
    joblib.delayed(add_numbers)(i, i+1) for i in range(5)
]

print("\nExecuting delayed tasks in parallel:")
parallel_results = joblib.Parallel(n_jobs=2)(delayed_tasks)
print(f"Parallel results: {parallel_results}")

# Execute individual delayed objects
single_result = joblib.Parallel(n_jobs=1)([joblib.delayed(multiply_numbers)(6, 7)])
print(f"Single delayed execution: {single_result[0]}")

# Direct function calls vs delayed execution
direct_result = add_numbers(10, 20)
delayed_result = joblib.Parallel(n_jobs=1)([joblib.delayed(add_numbers)(10, 20)])[0]
print(f"Direct: {direct_result}, Delayed: {delayed_result}")

print("Example 4 completed successfully\n")

## Example 5: joblib functionality test 5

In [ ]:
import joblib
import numpy as np
import tempfile
import os

# Example 5: hash function and utility functions
print("Example 5: joblib.hash and Utility Functions")

# Test hash function for reproducible hashing
data1 = {'numbers': [1, 2, 3], 'text': 'hello'}
data2 = {'numbers': [1, 2, 3], 'text': 'hello'}
data3 = {'numbers': [1, 2, 4], 'text': 'hello'}

hash1 = joblib.hash(data1)
hash2 = joblib.hash(data2)
hash3 = joblib.hash(data3)

print(f"Hash of data1: {hash1}")
print(f"Hash of data2: {hash2}")
print(f"Hash of data3: {hash3}")
print(f"data1 == data2 hashes: {hash1 == hash2}")
print(f"data1 == data3 hashes: {hash1 == hash3}")

# Hash numpy arrays
array1 = np.array([1, 2, 3, 4, 5])
array2 = np.array([1, 2, 3, 4, 5])
array3 = np.array([1, 2, 3, 4, 6])

array_hash1 = joblib.hash(array1)
array_hash2 = joblib.hash(array2)
array_hash3 = joblib.hash(array3)

print(f"\nArray hash1: {array_hash1[:16]}...")
print(f"Array hash2: {array_hash2[:16]}...")
print(f"Array hashes match: {array_hash1 == array_hash2}")
print(f"Different array hash: {array_hash1 == array_hash3}")

# Test effective_n_jobs function
n_jobs_test = joblib.effective_n_jobs(-1)
print(f"\nEffective n_jobs for -1: {n_jobs_test}")

n_jobs_test2 = joblib.effective_n_jobs(2)
print(f"Effective n_jobs for 2: {n_jobs_test2}")

# CPU count
try:
    cpu_count = joblib.cpu_count()
    print(f"CPU count: {cpu_count}")
except:
    print("CPU count not available")
    raise

# Test with different hashable data types
string_hash = joblib.hash("test string")
print(f"String hash: {string_hash[:16]}...")

# Complex object hashing with hashable types only
complex_obj = {
    'array': np.random.random(10),
    'list': list(range(5)),
    'tuple': (1, 2, 'test'),
    'string': 'hashable_string'
}
complex_hash = joblib.hash(complex_obj)
print(f"Complex object hash: {complex_hash[:16]}...")

print("Example 5 completed successfully\n")